# 📊 Phase 2 — EDA : Exploration et Analyse des Données

**Objectif de ce notebook :**
- Profiler en détail les données réelles (`FACT_ACTIVITE`)
- Détecter valeurs manquantes, anomalies, outliers
- Comprendre les distributions et patterns business
- Formuler des hypothèses pour les modèles ML

---
> 💡 **Pré-requis :** Avoir exécuté Phase 1 pour avoir le fichier `data/raw/fact_activite.csv`

## 🔧 Étape 1 — Chargement des données

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os

matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='husl')

# --- Charger les données depuis CSV (sauvegardé en Phase 1) ---
CSV_PATH = r'c:\4_ERP_BI\Semestre_2\E-commerce\ML_Engineering\data\raw\fact_activite.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
    print(f'✅ Données chargées depuis CSV : {df.shape[0]} lignes × {df.shape[1]} colonnes')
else:
    print('⚠️ CSV non trouvé. Connexion directe SQL Server...')
    from sqlalchemy import create_engine, text
    import urllib
    conn_str = (
        'DRIVER={ODBC Driver 17 for SQL Server};'
        'SERVER=GEORGE\\MSSQLSERVER03;'
        'DATABASE=DWH_ARTISANAT;'
        'Trusted_Connection=yes;'
    )
    params = urllib.parse.quote_plus(conn_str)
    engine = create_engine(f'mssql+pyodbc:///?odbc_connect={params}')
    query = """
    SELECT fa.*, dd.Annee, dd.Mois, dd.Trimestre, dd.Lib_Mois
    FROM FACT_ACTIVITE fa
    LEFT JOIN DIM_DATE dd ON fa.FK_DATE = dd.DATE_PK
    ORDER BY dd.Annee DESC, dd.Mois DESC
    """
    df = pd.read_sql(query, engine)
    print(f'✅ Données chargées depuis SQL : {df.shape[0]} lignes × {df.shape[1]} colonnes')

print('\nAperçu :')
display(df.head(3))

✅ Données chargées depuis CSV : 659 lignes × 23 colonnes

Aperçu :


,FK_Produit,FK_Client,FK_Date,FK_Geographie,FK_Fournisseur,FK_Concurrent,FK_SocialMedia,FK_Document,FK_Commande,FK_Type,...,Montant_TVA,Montant_TTC,Remise,Prix_concurrent,Likes,Commentaires,Annee,Mois,Trimestre,Lib_Mois
0,-1,237,20260304,-1,-1,-1,-1,290,-1,1,...,0.0,179.500000,0.0,0.0,0,0,2026,3,T1,March
1,-1,3,20260312,-1,-1,-1,-1,308,-1,1,...,0.0,17.365927,0.0,0.0,0,0,2026,3,T1,March
2,-1,-1,20260204,-1,-1,-1,8,-1,-1,3,...,0.0,0.000000,0.0,0.0,12,0,2026,2,T1,February


---
## 🔍 Étape 2 — Profil général du dataset

Comprendre la structure + types de données avant d'aller plus loin.

In [ ]:
print('=== INFORMATIONS GÉNÉRALES ===')
print(f'Nombre de lignes    : {df.shape[0]:,}')
print(f'Nombre de colonnes  : {df.shape[1]}')
print(f'Mémoire utilisée    : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

print('\n=== TYPES DE COLONNES ===')
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f'  {str(dtype):<12} : {count} colonnes')

print('\n=== DÉTAIL DES COLONNES ===')
info_df = pd.DataFrame({
    'Type': df.dtypes,
    'Non-Null': df.notna().sum(),
    'Null': df.isna().sum(),
    'Null %': (df.isna().sum() / len(df) * 100).round(1),
    'Unique': df.nunique(),
    'Sample': df.iloc[0]
})
display(info_df)

In [ ]:
# Statistiques descriptives pour les colonnes numériques
print('=== STATISTIQUES DESCRIPTIVES ===')
display(df.describe().round(2))

---
## 🚩 Étape 3 — Analyse des Valeurs Manquantes

Les valeurs manquantes peuvent biaiser les modèles ML. On doit les identifier et décider de la stratégie.

In [ ]:
# --- Calcul des manquants ---
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Percent', ascending=False)

if len(missing_df) == 0:
    print('✅ Aucune valeur manquante (NaN) détectée !')
    print('\n📌 Note : Vérifier aussi les valeurs sentinelles (-1 = inconnu dans un Data Warehouse)')
else:
    print(f'⚠️  {len(missing_df)} colonnes avec des valeurs manquantes :')
    display(missing_df)
    
    # Visualisation
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['Percent'].plot(kind='bar', ax=ax, color='#FF6B6B', edgecolor='black')
    ax.set_title('% de valeurs manquantes par colonne', fontweight='bold')
    ax.set_ylabel('% manquant')
    ax.axhline(y=50, color='red', linestyle='--', label='Seuil 50%')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Analyse des valeurs sentinelles (-1 = inconnu dans un DWH) ---
print('=== VALEURS SENTINELLES (-1 = inconnu dans ce Data Warehouse) ===')
fk_cols = [c for c in df.columns if c.startswith('FK_')]
print(f'\nColonnes FK détectées : {fk_cols}')
print()

sentinel_stats = []
for col in fk_cols:
    n_sentinels = (df[col] == -1).sum()
    pct = n_sentinels / len(df) * 100
    sentinel_stats.append({'Colonne': col, 'Valeur=-1': n_sentinels, '%': round(pct, 1)})

sentinel_df = pd.DataFrame(sentinel_stats).sort_values('%', ascending=False)
display(sentinel_df)

print('\n📌 Interprétation :')
print('  - FK avec beaucoup de -1 → dimension pas toujours applicable')
print('  - Ex: FK_SocialMedia = -1 pour les ventes non liées aux réseaux sociaux')

---
## 📈 Étape 4 — Distribution des Variables Numériques

Comprendre comment sont distribuées les valeurs → détection d'outliers, skewness.

In [ ]:
# Identifier les colonnes numériques pertinentes (pas les FK)
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns 
                if not c.startswith('FK_') and c not in ['Annee', 'Mois']]

print(f'Colonnes numériques analysées : {numeric_cols}')

if len(numeric_cols) > 0:
    n_cols = min(len(numeric_cols), 6)
    n_rows_plot = (n_cols + 2) // 3
    
    fig, axes = plt.subplots(n_rows_plot, 3, figsize=(15, 4 * n_rows_plot))
    axes = axes.flatten() if n_rows_plot > 1 else [axes] if n_cols == 1 else axes
    
    for i, col in enumerate(numeric_cols[:6]):
        ax = axes[i] if n_cols > 1 else axes
        data = df[col].dropna()
        
        ax.hist(data, bins=20, color='#4ECDC4', edgecolor='black', alpha=0.8)
        ax.axvline(data.mean(), color='red', linestyle='--', label=f'Moy={data.mean():.0f}')
        ax.axvline(data.median(), color='orange', linestyle='--', label=f'Med={data.median():.0f}')
        ax.set_title(col, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    # Masquer les axes vides
    for j in range(n_cols, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Distribution des variables numériques', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Skewness (asymétrie) et Kurtosis ---
print('=== ASYMÉTRIE DES DISTRIBUTIONS ===')
skew_df = pd.DataFrame({
    'Skewness': df[numeric_cols].skew().round(2),
    'Kurtosis': df[numeric_cols].kurtosis().round(2),
    'Min': df[numeric_cols].min().round(2),
    'Max': df[numeric_cols].max().round(2),
    'Moyenne': df[numeric_cols].mean().round(2)
}).sort_values('Skewness', ascending=False)

display(skew_df)

print('\n📌 Interprétation Skewness :')
print('  - |Skewness| < 0.5 : Distribution symétrique ✅')
print('  - 0.5 < |Skewness| < 1 : Légèrement asymétrique ⚠️')
print('  - |Skewness| > 1 : Très asymétrique → transformation log recommandée 🔄')

---
## 🎯 Étape 5 — Analyse du Chiffre d'Affaires (Montant_TTC)

La variable cible principale pour nos modèles ML.

In [ ]:
# Vérifier si la colonne existe
if 'Montant_TTC' in df.columns:
    ca = df['Montant_TTC']
    ca_positif = ca[ca > 0]  # Exclure les 0 (ex: posts social media)
    
    print('=== ANALYSE DU MONTANT_TTC ===')
    print(f'Total lignes         : {len(ca):,}')
    print(f'Lignes avec CA > 0   : {len(ca_positif):,} ({len(ca_positif)/len(ca)*100:.1f}%)')
    print(f'Lignes avec CA = 0   : {(ca == 0).sum():,} ({(ca == 0).sum()/len(ca)*100:.1f}%)')
    print(f'\nStatistiques (CA > 0) :')
    print(f'  Total CA    : {ca_positif.sum():,.0f} DT')
    print(f'  Moyenne     : {ca_positif.mean():,.2f} DT')
    print(f'  Médiane     : {ca_positif.median():,.2f} DT')
    print(f'  Min         : {ca_positif.min():,.2f} DT')
    print(f'  Max         : {ca_positif.max():,.2f} DT')
    print(f'  Écart-type  : {ca_positif.std():,.2f} DT')
else:
    ca_col = [c for c in df.columns if 'montant' in c.lower() or 'ca' in c.lower() or 'vente' in c.lower()]
    print(f'⚠️ Colonne Montant_TTC non trouvée. Colonnes similaires : {ca_col}')

In [ ]:
if 'Montant_TTC' in df.columns:
    df_ventes = df[df['Montant_TTC'] > 0].copy()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Distribution brute
    axes[0].hist(df_ventes['Montant_TTC'], bins=30, color='#4ECDC4', edgecolor='black', alpha=0.8)
    axes[0].set_title('Distribution brute', fontweight='bold')
    axes[0].set_xlabel('Montant TTC (DT)')
    axes[0].set_ylabel('Fréquence')
    
    # 2. Distribution log (pour réduire l'asymétrie)
    log_ca = np.log1p(df_ventes['Montant_TTC'])
    axes[1].hist(log_ca, bins=30, color='#45B7D1', edgecolor='black', alpha=0.8)
    axes[1].set_title('Distribution log(CA+1)', fontweight='bold')
    axes[1].set_xlabel('log(Montant TTC + 1)')
    
    # 3. Boxplot
    axes[2].boxplot(df_ventes['Montant_TTC'], vert=True)
    axes[2].set_title('Boxplot (avec outliers)', fontweight='bold')
    axes[2].set_ylabel('Montant TTC (DT)')
    
    plt.suptitle('💰 Analyse du Chiffre d\'Affaires', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(r'c:\4_ERP_BI\Semestre_2\E-commerce\ML_Engineering\reports\distribution_ca.png', dpi=150)
    plt.show()
    print('✅ Graphique sauvegardé')

---
## 🔎 Étape 6 — Détection des Outliers

Les outliers peuvent fausser les modèles. On utilise la méthode IQR (Interquartile Range).

In [ ]:
def detect_outliers_iqr(series, col_name):
    """Détecte les outliers par méthode IQR"""
    s = series.dropna()
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = s[(s < lower) | (s > upper)]
    return {
        'Colonne': col_name,
        'Q1': round(Q1, 2), 'Q3': round(Q3, 2), 'IQR': round(IQR, 2),
        'Borne_inf': round(lower, 2), 'Borne_sup': round(upper, 2),
        'Nb_outliers': len(outliers),
        '%_outliers': round(len(outliers) / len(s) * 100, 1)
    }

print('=== DÉTECTION DES OUTLIERS (méthode IQR) ===')
outlier_results = []
for col in numeric_cols:
    if df[col].std() > 0:  # Ignorer les colonnes constantes
        result = detect_outliers_iqr(df[col], col)
        outlier_results.append(result)

outlier_df = pd.DataFrame(outlier_results).sort_values('%_outliers', ascending=False)
display(outlier_df)

print('\n📌 Stratégies de traitement des outliers :')
print('  - < 5% outliers → garder (bruit naturel)')
print('  - 5-15% outliers → winsorisation (cap & floor)')
print('  - > 15% outliers → vérifier la logique métier')

In [ ]:
# Visualisation boxplots pour comparer
if len(numeric_cols) > 0:
    fig, axes = plt.subplots(1, min(len(numeric_cols), 4), figsize=(14, 5))
    if len(numeric_cols) == 1:
        axes = [axes]
    
    for i, col in enumerate(numeric_cols[:4]):
        ax = axes[i]
        data = df[col].dropna()
        if data.std() > 0:
            ax.boxplot(data)
            ax.set_title(col, fontweight='bold')
            ax.set_ylabel('Valeur')
    
    plt.suptitle('🎯 Boxplots — Détection des Outliers', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 📅 Étape 7 — Analyse Temporelle

Comprendre l'évolution dans le temps → identifier des tendances et saisonnalités.

In [ ]:
time_cols = [c for c in ['Annee', 'Mois', 'Trimestre', 'Lib_Mois'] if c in df.columns]
print(f'Colonnes temporelles disponibles : {time_cols}')

if 'Mois' in df.columns:
    # Distribution par mois
    print('\n=== DISTRIBUTION PAR MOIS ===')
    monthly = df.groupby('Mois').size().rename('Nb Transactions')
    display(monthly.to_frame().T)

if 'Annee' in df.columns:
    print('\n=== DISTRIBUTION PAR ANNÉE ===')
    yearly = df.groupby('Annee').size().rename('Nb Transactions')
    display(yearly.to_frame())

In [ ]:
if 'Montant_TTC' in df.columns and 'Mois' in df.columns:
    df_ventes = df[df['Montant_TTC'] > 0].copy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # CA par mois
    ca_mois = df_ventes.groupby('Mois')['Montant_TTC'].agg(['sum', 'mean', 'count'])
    axes[0].bar(ca_mois.index, ca_mois['sum'], color='#4ECDC4', edgecolor='black')
    axes[0].set_title('CA Total par Mois', fontweight='bold')
    axes[0].set_xlabel('Mois')
    axes[0].set_ylabel('CA Total (DT)')
    axes[0].set_xticks(ca_mois.index)
    axes[0].grid(alpha=0.3, axis='y')
    
    # Nb transactions par mois
    axes[1].bar(ca_mois.index, ca_mois['count'], color='#45B7D1', edgecolor='black')
    axes[1].set_title('Nb Transactions par Mois', fontweight='bold')
    axes[1].set_xlabel('Mois')
    axes[1].set_ylabel('Nombre transactions')
    axes[1].set_xticks(ca_mois.index)
    axes[1].grid(alpha=0.3, axis='y')
    
    plt.suptitle('📅 Analyse Temporelle des Ventes', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(r'c:\4_ERP_BI\Semestre_2\E-commerce\ML_Engineering\reports\analyse_temporelle.png', dpi=150)
    plt.show()
    
    print('\nCA par mois (DT) :')
    display(ca_mois.rename(columns={'sum':'CA Total', 'mean':'CA Moyen', 'count':'Nb Trans.'}).round(2))

---
## 🌐 Étape 8 — Analyse des Réseaux Sociaux (Likes & Commentaires)

Comprendre l'impact du engagement social sur les ventes.

In [ ]:
social_cols = [c for c in ['Likes', 'Commentaires', 'FK_SocialMedia'] if c in df.columns]
print(f'Colonnes Social Media disponibles : {social_cols}')

if 'Likes' in df.columns:
    likes = df['Likes']
    print(f'\n=== ANALYSE DES LIKES ===')
    print(f'Total Likes         : {likes.sum():,}')
    print(f'Posts avec Likes    : {(likes > 0).sum():,} ({(likes > 0).sum()/len(likes)*100:.1f}%)')
    print(f'Posts sans Likes    : {(likes == 0).sum():,} ({(likes == 0).sum()/len(likes)*100:.1f}%)')
    print(f'Moyenne (posts actifs) : {likes[likes > 0].mean():.1f} likes')
    print(f'Max Likes           : {likes.max():,}')

if 'Commentaires' in df.columns:
    comms = df['Commentaires']
    print(f'\n=== ANALYSE DES COMMENTAIRES ===')
    print(f'Total Commentaires  : {comms.sum():,}')
    print(f'Posts avec Comms    : {(comms > 0).sum():,} ({(comms > 0).sum()/len(comms)*100:.1f}%)')

In [ ]:
# Corrélation Likes ↔ Ventes
if 'Likes' in df.columns and 'Montant_TTC' in df.columns:
    df_social = df[df['Likes'] > 0].copy()
    
    if len(df_social) > 0:
        print(f'Posts Social Media avec Likes : {len(df_social)}')
        
        # Répartition par réseau social si disponible
        if 'FK_SocialMedia' in df.columns:
            print('\n=== RÉPARTITION PAR RÉSEAU SOCIAL ===')
            social_dist = df.groupby('FK_SocialMedia').agg(
                Nb_posts=('FK_SocialMedia', 'count'),
                Total_Likes=('Likes', 'sum'),
                Moy_Likes=('Likes', 'mean')
            ).round(1)
            display(social_dist[social_dist.index != -1])
    else:
        print('ℹ️  Pas de données Social Media avec Likes dans ce dataset')

---
## 🔗 Étape 9 — Matrice de Corrélation

Identifier les relations entre variables numériques → features importantes pour le ML.

In [ ]:
# Colonnes numériques pour la corrélation
corr_cols = [c for c in df.select_dtypes(include=[np.number]).columns 
             if not c.startswith('FK_')]

if len(corr_cols) >= 2:
    corr_matrix = df[corr_cols].corr()
    
    fig_size = max(8, len(corr_cols) * 0.8)
    plt.figure(figsize=(fig_size, fig_size * 0.8))
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Masque triangle sup
    sns.heatmap(corr_matrix, 
                annot=True, fmt='.2f', 
                cmap='coolwarm',
                square=True, 
                mask=mask,
                linewidths=0.5,
                vmin=-1, vmax=1,
                annot_kws={'size': 9})
    
    plt.title('🔗 Matrice de Corrélation (variables numériques)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(r'c:\4_ERP_BI\Semestre_2\E-commerce\ML_Engineering\reports\matrice_correlation.png', dpi=150)
    plt.show()
    print('✅ Matrice sauvegardée')
    
    # Top corrélations avec Montant_TTC
    if 'Montant_TTC' in corr_cols:
        print('\n=== TOP CORRÉLATIONS AVEC MONTANT_TTC ===')
        corr_with_ca = corr_matrix['Montant_TTC'].drop('Montant_TTC').sort_values(key=abs, ascending=False)
        display(corr_with_ca.to_frame().rename(columns={'Montant_TTC': 'Corrélation'}).round(3))

---
## 📊 Étape 10 — Analyse par Type de Transaction

Comprendre les différents types d'activité dans FACT_ACTIVITE.

In [ ]:
if 'FK_Type' in df.columns:
    print('=== RÉPARTITION PAR TYPE D\'ACTIVITÉ ===')
    type_dist = df.groupby('FK_Type').agg(
        Nb_Lignes=('FK_Type', 'count'),
        CA_Total=('Montant_TTC', 'sum') if 'Montant_TTC' in df.columns else ('FK_Type', 'count'),
        CA_Moyen=('Montant_TTC', 'mean') if 'Montant_TTC' in df.columns else ('FK_Type', 'count'),
        Total_Likes=('Likes', 'sum') if 'Likes' in df.columns else ('FK_Type', 'count')
    ).round(2)
    
    type_dist['%_Lignes'] = (type_dist['Nb_Lignes'] / len(df) * 100).round(1)
    display(type_dist)
    
    print('\n📌 Décodage des types :')
    print('  Type 1 = Transaction de vente (Montant_TTC > 0)')
    print('  Type 2 = Achat/Dépense (Montant_TTC négatif)')
    print('  Type 3 = Activité Social Media (Likes/Commentaires)')

In [ ]:
if 'FK_Type' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart répartition
    type_counts = df['FK_Type'].value_counts()
    type_labels = [f'Type {t}' for t in type_counts.index]
    axes[0].pie(type_counts.values, labels=type_labels, autopct='%1.1f%%',
                colors=['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4'])
    axes[0].set_title('Répartition par Type', fontweight='bold')
    
    # CA par type (si disponible)
    if 'Montant_TTC' in df.columns:
        ca_by_type = df.groupby('FK_Type')['Montant_TTC'].sum()
        axes[1].bar([f'Type {t}' for t in ca_by_type.index], ca_by_type.values,
                    color=['#4ECDC4', '#FF6B6B', '#45B7D1'][:len(ca_by_type)],
                    edgecolor='black')
        axes[1].set_title('CA Total par Type (DT)', fontweight='bold')
        axes[1].set_ylabel('Montant TTC (DT)')
        axes[1].grid(alpha=0.3, axis='y')
    
    plt.suptitle('📊 Analyse par Type d\'Activité', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 🧹 Étape 11 — Préparation du Dataset ML

Créer un dataset propre et enrichi, prêt pour les modèles ML.

In [ ]:
print('=== PRÉPARATION DU DATASET ML ===')

# 1. Filtrer uniquement les transactions de vente
if 'FK_Type' in df.columns:
    df_ml = df[df['FK_Type'] == 1].copy()
    print(f'1. Filtrage type=1 (ventes) : {len(df):,} → {len(df_ml):,} lignes')
else:
    df_ml = df[df.get('Montant_TTC', pd.Series([0])) > 0].copy()
    print(f'1. Filtrage CA > 0 : {len(df):,} → {len(df_ml):,} lignes')

# 2. Feature engineering — variables dérivées
if 'Montant_TTC' in df_ml.columns and 'Montant_TVA' in df_ml.columns:
    df_ml['Montant_HT'] = df_ml['Montant_TTC'] - df_ml.get('Montant_TVA', 0)
    print('2. Calculé Montant_HT = TTC - TVA ✅')

if 'Remise' in df_ml.columns:
    df_ml['A_Remise'] = (df_ml['Remise'] > 0).astype(int)
    print('3. Créé feature binaire A_Remise ✅')

if 'Likes' in df_ml.columns:
    df_ml['A_Social'] = (df_ml['FK_SocialMedia'] != -1).astype(int) if 'FK_SocialMedia' in df_ml.columns else 0
    print('4. Créé A_Social (lié à réseau social) ✅')

if 'Mois' in df_ml.columns:
    df_ml['Est_WeekEnd'] = 0  # Placeholder (FK_Date → à enrichir)
    # Trimestre déjà disponible
    print('5. Variables temporelles disponibles ✅')

print(f'\n✅ Dataset ML final : {df_ml.shape[0]} lignes × {df_ml.shape[1]} colonnes')
display(df_ml.head(3))

In [ ]:
# Sauvegarder le dataset processed
PROCESSED_PATH = r'c:\4_ERP_BI\Semestre_2\E-commerce\ML_Engineering\data\processed\dataset_ml.csv'
df_ml.to_csv(PROCESSED_PATH, index=False, encoding='utf-8-sig')
print(f'✅ Dataset ML sauvegardé → {PROCESSED_PATH}')
print(f'   {df_ml.shape[0]} lignes × {df_ml.shape[1]} colonnes')

---
## 💡 Étape 12 — Synthèse EDA et Hypothèses ML

Résumé des découvertes et hypothèses pour les prochaines phases.

In [ ]:
print("""╔══════════════════════════════════════════════════════════════╗
║          📊 SYNTHÈSE EDA — FACT_ACTIVITE                     ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ✅ STRUCTURE DES DONNÉES                                    ║
║  • Dataset : 659 lignes × 23 colonnes                        ║
║  • Types d'activité : Ventes, Achats, Social Media           ║
║  • Données temporelles : Mois, Trimestre, Année              ║
║                                                              ║
║  ✅ QUALITÉ DES DONNÉES                                      ║
║  • Valeurs manquantes (NaN) : vérifiées                      ║
║  • Valeurs sentinelles (-1) : normales en DWH                ║
║  • Outliers : identifiés et quantifiés                       ║
║                                                              ║
║  🎯 HYPOTHÈSES POUR LE ML                                    ║
║  1. Prédire le CA (Montant_TTC) = Régression                 ║
║  2. Classifier si vente → Social Media = Classification      ║
║  3. Détection d'anomalies dans les montants                  ║
║  4. Clustering des produits par profil de vente              ║
║                                                              ║
║  🚀 PROCHAINE ÉTAPE → Phase 3 : Feature Engineering         ║
╚══════════════════════════════════════════════════════════════╝""")

In [ ]:
# Rapport automatique
if 'Montant_TTC' in df.columns:
    df_ventes = df[df.get('Montant_TTC', pd.Series([0])) > 0]
    
    print('📈 RAPPORT EDA AUTOMATIQUE')
    print('=' * 50)
    print(f'Dataset brut       : {len(df):,} lignes')
    print(f'Lignes de ventes   : {len(df_ventes):,} ({len(df_ventes)/len(df)*100:.0f}%)')
    print(f'CA Total           : {df_ventes["Montant_TTC"].sum():,.2f} DT')
    print(f'CA Moyen/vente     : {df_ventes["Montant_TTC"].mean():,.2f} DT')
    
    if 'Mois' in df.columns:
        best_month = df_ventes.groupby('Mois')['Montant_TTC'].sum().idxmax()
        print(f'Meilleur mois CA   : Mois n°{best_month}')
    
    if 'Likes' in df.columns:
        print(f'Total Likes SM     : {df["Likes"].sum():,}')
    
    print('\n✅ EDA Phase 2 terminée. Dataset ML prêt.')
    print('→ Continuer avec Phase 3 : Feature Engineering')

---
## ✅ Récapitulatif — Phase 2

| Étape | Tâche | Résultat |
|-------|-------|----------|
| 1 | Chargement des données | `FACT_ACTIVITE` chargé depuis CSV/SQL |
| 2 | Profil général | Types, tailles, mémoire |
| 3 | Valeurs manquantes | NaN + sentinelles (-1) |
| 4 | Distributions | Histogrammes, skewness |
| 5 | Analyse CA | Distribution, outliers |
| 6 | Détection outliers | IQR, boxplots |
| 7 | Analyse temporelle | Par mois, trimestre |
| 8 | Social Media | Likes, commentaires |
| 9 | Corrélations | Matrice, top features |
| 10 | Types d'activité | Ventes vs Social |
| 11 | Préparation | Dataset ML sauvegardé |
| 12 | Synthèse | Hypothèses formulées |

## 🚀 Prochaine étape — Phase 3 : Feature Engineering

Dans `03_Feature_Engineering.ipynb` :
- Création de nouvelles features (lag, rolling, ratios)
- Encodage des variables catégorielles
- Normalisation et standardisation
- Sélection des features importantes